# AegisRisk — Notebook 01: Credit Risk Scorecard & Survival Analysis

**Dataset:** Home Credit Default Risk (`application_train.csv`, `bureau.csv`, `previous_application.csv`, `installments_payments.csv`)

**Objectives:**
1. Engineer relational credit & behavioral ratios and save a shared Parquet checkpoint to Google Drive.
2. Perform Weight of Evidence (WoE) binning & Information Value (IV) feature selection (`optbinning`).
3. Train a calibrated Logistic Regression Credit Scorecard (300–850 PDO scale).
4. Validate with banking metrics (ROC-AUC, Gini, KS-Statistic, Brier Score, Score PSI, Decile Lift).
5. Fit a Cox Proportional Hazards (`lifelines`) survival model for time-to-default and compute Expected Loss (`EL = PD × LGD × EAD`).
6. Export frozen artifacts and `credit_schema.json` to Google Drive for downstream notebooks.

In [ ]:
# Cell 1: Mount Google Drive, Create Folders & Install Dependencies
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/aegis-risk')
RAW_DATA_DIR = Path('/content/data/raw')  # Local SSD for fast I/O
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
NB01_DIR = BASE_DIR / 'artifacts' / 'nb01_scorecard'
ARTIFACTS_ROOT = BASE_DIR / 'artifacts'

for d in [RAW_DATA_DIR, PROCESSED_DIR, NB01_DIR, ARTIFACTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

!pip install -q optbinning lifelines pyarrow kaggle
print("Environment ready. Artifacts will save to:", NB01_DIR)

In [ ]:
# Cell 2: Download Home Credit Dataset via Kaggle API
import os
from google.colab import files

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception:
        print("Upload your kaggle.json file:")
        uploaded = files.upload()
        for fn, content in uploaded.items():
            kaggle_json.write_bytes(content)
        os.chmod(kaggle_json, 0o600)

if not (RAW_DATA_DIR / 'application_train.csv').exists():
    !kaggle competitions download -c home-credit-default-risk -p /content/data/raw
    !unzip -q -o /content/data/raw/home-credit-default-risk.zip -d /content/data/raw
    !rm -f /content/data/raw/home-credit-default-risk.zip

print("Downloaded files:", [p.name for p in sorted(RAW_DATA_DIR.glob('*.csv'))])

In [ ]:
# Cell 3: Memory-Safe Loader & Core Application Feature Engineering
import gc
import numpy as np
import pandas as pd

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and str(col_type) != 'category':
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type).startswith('int'):
                if c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

app = pd.read_csv(RAW_DATA_DIR / 'application_train.csv')
app = reduce_mem_usage(app)

# Fix known 365243 pensioner anomaly in DAYS_EMPLOYED
app['DAYS_EMPLOYED_ANOMALY'] = (app['DAYS_EMPLOYED'] == 365243).astype(np.int8)
app['DAYS_EMPLOYED'] = app['DAYS_EMPLOYED'].replace(365243, np.nan)

# Core underwriting financial ratios
app['AGE_YEARS'] = (-app['DAYS_BIRTH'] / 365.25).astype(np.float32)
app['YEARS_EMPLOYED'] = (-app['DAYS_EMPLOYED'] / 365.25).astype(np.float32)
app['DAYS_EMPLOYED_RATIO'] = (app['DAYS_EMPLOYED'] / app['DAYS_BIRTH']).astype(np.float32)
app['CREDIT_TO_INCOME_RATIO'] = (app['AMT_CREDIT'] / (app['AMT_INCOME_TOTAL'] + 1.0)).astype(np.float32)
app['ANNUITY_TO_INCOME_RATIO'] = (app['AMT_ANNUITY'] / (app['AMT_INCOME_TOTAL'] + 1.0)).astype(np.float32)
app['CREDIT_TERM_MONTHS'] = (app['AMT_CREDIT'] / (app['AMT_ANNUITY'] + 1.0)).astype(np.float32)
app['GOODS_TO_CREDIT_RATIO'] = (app['AMT_GOODS_PRICE'] / (app['AMT_CREDIT'] + 1.0)).astype(np.float32)

ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
app['EXT_SOURCE_MEAN'] = app[ext_cols].mean(axis=1).astype(np.float32)
app['EXT_SOURCE_STD'] = app[ext_cols].std(axis=1).astype(np.float32)

print(f"Base application shape: {app.shape} | Default rate: {app['TARGET'].mean():.4%}")

In [ ]:
# Cell 4: Relational Aggregations (Bureau, Previous Apps & Installments)

# 1. Credit Bureau Aggregations
bureau = reduce_mem_usage(pd.read_csv(RAW_DATA_DIR / 'bureau.csv'))
bureau['IS_ACTIVE'] = (bureau['CREDIT_ACTIVE'] == 'Active').astype(np.int8)

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('IS_ACTIVE', 'sum'),
    BUREAU_TOTAL_CREDIT=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_MAX_OVERDUE_DAYS=('CREDIT_DAY_OVERDUE', 'max')
).reset_index()

bureau_agg['BUREAU_ACTIVE_DEBT_RATIO'] = (
    bureau_agg['BUREAU_TOTAL_DEBT'] / (bureau_agg['BUREAU_TOTAL_CREDIT'] + 1.0)
).astype(np.float32)

app = app.merge(bureau_agg, on='SK_ID_CURR', how='left')
del bureau, bureau_agg
gc.collect()

# 2. Previous Applications Aggregations
prev = reduce_mem_usage(pd.read_csv(RAW_DATA_DIR / 'previous_application.csv'))
prev['IS_REFUSED'] = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(np.int8)
prev['IS_APPROVED'] = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(np.int8)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_REFUSAL_RATE=('IS_REFUSED', 'mean'),
    PREV_APPROVAL_RATE=('IS_APPROVED', 'mean'),
    PREV_MEAN_CREDIT=('AMT_CREDIT', 'mean')
).reset_index()

app = app.merge(prev_agg, on='SK_ID_CURR', how='left')
del prev, prev_agg
gc.collect()

# 3. Installments Payments Aggregations (Behavior & Duration Proxy)
inst = reduce_mem_usage(pd.read_csv(RAW_DATA_DIR / 'installments_payments.csv'))
inst['DAYS_LATE'] = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
inst['IS_LATE'] = (inst['DAYS_LATE'] > 0).astype(np.int8)
inst['PAYMENT_RATIO'] = (inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1.0)).astype(np.float32)

inst_agg = inst.groupby('SK_ID_CURR').agg(
    INSTALLMENT_COUNT=('NUM_INSTALMENT_NUMBER', 'count'),
    INSTALLMENT_LATE_PAYMENT_RATIO=('IS_LATE', 'mean'),
    INSTALLMENT_MAX_DAYS_LATE=('DAYS_LATE', 'max'),
    INSTALLMENT_MEAN_PAYMENT_RATIO=('PAYMENT_RATIO', 'mean'),
    OBS_HISTORY_DAYS=('DAYS_INSTALMENT', lambda x: float(x.max() - x.min()))
).reset_index()

app = app.merge(inst_agg, on='SK_ID_CURR', how='left')
del inst, inst_agg
gc.collect()

print(f"Merged feature table shape: {app.shape}")

In [ ]:
# Cell 5: Save Shared Parquet Checkpoint & Stratified Train/Test Split
from sklearn.model_selection import train_test_split

# Save full engineered table for NB 02 (XGBoost Challenger)
parquet_path = PROCESSED_DIR / 'hc_features_train.parquet'
app.to_parquet(parquet_path, index=False)
print(f"Saved shared feature checkpoint to: {parquet_path}")

# Stratified 80/20 split (fixed random_state=42 across all notebooks)
y = app['TARGET'].astype(int)
X = app.drop(columns=['TARGET'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")

In [ ]:
# Cell 6: WoE/IV Feature Selection & PDO Credit Scorecard (300-850 Scale)
from optbinning import BinningProcess, Scorecard
from sklearn.linear_model import LogisticRegression

feature_cols = [c for c in X_train.columns if c != 'SK_ID_CURR']
cat_cols = X_train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

binning_process = BinningProcess(
    variable_names=feature_cols,
    categorical_variables=cat_cols,
    min_prebin_size=0.05,
    selection_criteria={"iv": {"min": 0.02, "max": 0.60, "strategy": "highest"}}
)

estimator = LogisticRegression(C=0.1, solver='lbfgs', max_iter=1000)

# PDO Scaling: 600 points at 19:1 Good/Bad odds (PD = 5%), +20 points doubles the odds
scorecard = Scorecard(
    binning_process=binning_process,
    estimator=estimator,
    scaling_method="pdo_odds",
    scaling_method_params={"pdo": 20, "odds": 19, "scorecard_points": 600}
)

print("Fitting WoE BinningProcess and Logistic Scorecard...")
scorecard.fit(X_train[feature_cols], y_train)

selected_vars = scorecard.binning_process_.get_support(names=True)
print(f"Selected {len(selected_vars)} features with IV in [0.02, 0.60].")

summary_iv = scorecard.binning_process_.summary()
display(summary_iv[summary_iv['selected']].sort_values('iv', ascending=False).head(15))
display(scorecard.table(style="summary").head(20))

In [ ]:
# Cell 7: Banking Validation Suite (AUC, Gini, KS, PSI, Deciles & Gray-Zone)
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.stats import ks_2samp

pd_train = scorecard.predict_proba(X_train[feature_cols])[:, 1]
pd_test = scorecard.predict_proba(X_test[feature_cols])[:, 1]
scores_train = scorecard.score(X_train[feature_cols])
scores_test = scorecard.score(X_test[feature_cols])

auc = roc_auc_score(y_test, pd_test)
gini = 2 * auc - 1
ks_stat = ks_2samp(pd_test[y_test == 0], pd_test[y_test == 1]).statistic
brier = brier_score_loss(y_test, pd_test)

def calculate_psi(expected, actual, bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    exp_pct = np.clip(np.histogram(expected, breakpoints)[0] / len(expected), 1e-4, None)
    act_pct = np.clip(np.histogram(actual, breakpoints)[0] / len(actual), 1e-4, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

psi_score = calculate_psi(scores_train, scores_test)
print(f"ROC-AUC: {auc:.4f} | Gini: {gini:.4f} | KS-Stat: {ks_stat:.4f} | Brier: {brier:.4f} | Score PSI: {psi_score:.4f}")

# Decile Lift Table
eval_df = pd.DataFrame({'y_true': y_test.values, 'pd': pd_test, 'score': scores_test})
eval_df['decile'] = pd.qcut(eval_df['pd'], 10, labels=False, duplicates='drop') + 1
decile_table = eval_df.groupby('decile').agg(
    count=('y_true', 'count'),
    defaults=('y_true', 'sum'),
    actual_dr=('y_true', 'mean'),
    avg_pd=('pd', 'mean'),
    avg_score=('score', 'mean')
).sort_index(ascending=False)
display(decile_table)

# Gray-zone routing check for NB 06 LangGraph Committee
gray_mask = (eval_df['pd'] >= 0.05) & (eval_df['pd'] <= 0.18)
print(f"Auto-Approve (<5% PD): {(eval_df['pd'] < 0.05).mean():.2%} | "
      f"Gray-Zone Agents (5%-18% PD): {gray_mask.mean():.2%} | "
      f"Auto-Decline (>18% PD): {(eval_df['pd'] > 0.18).mean():.2%}")

In [ ]:
# Cell 8: Cox Survival Analysis (Time-to-Default) & Expected Loss (EL)
from lifelines import CoxPHFitter

surv_features = [
    'EXT_SOURCE_MEAN', 'CREDIT_TO_INCOME_RATIO', 'ANNUITY_TO_INCOME_RATIO',
    'AGE_YEARS', 'BUREAU_ACTIVE_DEBT_RATIO', 'PREV_REFUSAL_RATE',
    'INSTALLMENT_LATE_PAYMENT_RATIO'
]

surv_df = X_train[surv_features].copy()
surv_df = surv_df.fillna(surv_df.median())

# Use observed installment history (in months) or loan term as duration proxy
duration_months = (X_train['OBS_HISTORY_DAYS'].fillna(X_train['CREDIT_TERM_MONTHS'] * 30.0) / 30.0).clip(lower=1.0, upper=60.0)
surv_df['DURATION_MONTHS'] = duration_months.values
surv_df['TARGET'] = y_train.values

cph = CoxPHFitter(penalizer=0.1)
cph.fit(surv_df, duration_col='DURATION_MONTHS', event_col='TARGET')
print(f"Cox PH Concordance Index: {cph.concordance_index_:.4f}")
cph.print_summary(columns=['coef', 'exp(coef)', 'p'])

# Expected Loss = PD * LGD * EAD on test set sample
test_surv_in = X_test[surv_features].fillna(X_train[surv_features].median())
surv_probs_24m = cph.predict_survival_function(test_surv_in.iloc[:5], times=[12, 24, 36])

# LGD proxy: 1 - historical installment payment ratio (clipped to realistic [0.25, 0.85] unsecured band)
lgd_proxy = (1.0 - X_test['INSTALLMENT_MEAN_PAYMENT_RATIO'].fillna(0.55)).clip(0.25, 0.85)
ead_proxy = X_test['AMT_CREDIT'].values
expected_loss = pd_test * lgd_proxy.values * ead_proxy

el_preview = pd.DataFrame({
    'SK_ID_CURR': X_test['SK_ID_CURR'].values[:5],
    'Score_300_850': scores_test[:5].round(0),
    'PD_1Yr': pd_test[:5].round(4),
    'Cum_PD_24M_Cox': (1.0 - surv_probs_24m.loc[24].values).round(4),
    'LGD_Proxy': lgd_proxy.values[:5].round(2),
    'EAD_USD': ead_proxy[:5].round(0),
    'Expected_Loss_USD': expected_loss[:5].round(2)
})
display(el_preview)

In [ ]:
# Cell 9: Export Frozen Artifacts & Update Shared Registry Manifest
import pickle
import json
from datetime import datetime

# 1. Save Python model artifacts
with open(NB01_DIR / 'woe_binner.pkl', 'wb') as f:
    pickle.dump(scorecard.binning_process_, f)

with open(NB01_DIR / 'scorecard_lr.pkl', 'wb') as f:
    pickle.dump(scorecard, f)

with open(NB01_DIR / 'cox_survival.pkl', 'wb') as f:
    pickle.dump(cph, f)

# 2. Save NB 01 JSON Schema Contract
credit_schema = {
    "module": "nb01_credit_scorecard",
    "dataset": "home-credit-default-risk",
    "primary_key": "SK_ID_CURR",
    "target": "TARGET",
    "all_input_features": feature_cols,
    "selected_woe_features": list(selected_vars),
    "survival_features": surv_features,
    "scorecard_params": {
        "base_score": 600,
        "base_odds": 19.0,
        "pdo": 20,
        "auto_approve_pd_max": 0.05,
        "gray_zone_pd_range": [0.05, 0.18],
        "auto_decline_pd_min": 0.18
    },
    "validation_metrics": {
        "roc_auc": round(float(auc), 4),
        "gini": round(float(gini), 4),
        "ks_statistic": round(float(ks_stat), 4),
        "brier_score": round(float(brier), 4),
        "psi_score": round(float(psi_score), 4),
        "cox_concordance": round(float(cph.concordance_index_), 4)
    },
    "updated_at": datetime.utcnow().isoformat() + "Z"
}

with open(NB01_DIR / 'credit_schema.json', 'w') as f:
    json.dump(credit_schema, f, indent=2)

# 3. Update global registry_manifest.json
manifest_path = ARTIFACTS_ROOT / 'registry_manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest['nb01_scorecard'] = {
    "status": "ready",
    "path": str(NB01_DIR),
    "metrics": credit_schema["validation_metrics"]
}
manifest_path.write_text(json.dumps(manifest, indent=2))

print("All NB 01 artifacts exported successfully:", [p.name for p in sorted(NB01_DIR.iterdir())])